# Imports

In [1]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
api.start_spark(n_executors=400, config=config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


23/11/02 01:07:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
23/11/02 01:07:54 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
23/11/02 01:08:06 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [5]:
@F.udf("string")
def to_date(ts):
    return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

In [6]:
year0 = "2020"
year1 = str(int(year0) + 1)

In [7]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

In [8]:
df_airports_raw = (
    api.dataframe("ArincAirport", **dates, metadata=True)
    .select(
        F.col("identification.name").alias("icao_code"),
        F.col("identification.icao_region").alias("icao_region"),
       "iata_code",
       "full_name",
        "latitude",
        "longitude",
        "elevation",
        F.col("magnetic_variation.modeled").alias("magnetic_variation"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
    .withColumn("full_name", F.regexp_replace("full_name", ",", ""))
#    .filter(F.col("usage") == "PUBLIC")
#    .sort(F.desc("end_date"))
    .orderBy("icao_code", "icao_region")
)

#df_airports_raw.show()

Multiple versions found: 3.1.12.2, 3.1.22


23/11/02 01:08:44 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
23/11/02 01:08:59 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
23/11/02 01:09:14 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


In [9]:
#df_airports_raw.count()

In [10]:
window = Window.partitionBy("icao_code").orderBy(col("end_date").desc())

df_airports = (df_airports_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
)
 
#df_airports.show()

In [11]:
#df_airports.count()

In [12]:
(
    df_airports
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/airports", compression="None", mode="overwrite")
)